# Handwritten Text Recognition

See the README for setup and required inputs.

In [ ]:
import random
import re
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from Levenshtein import distance
from PIL import Image
from sklearn.metrics import accuracy_score
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, transforms
from torchvision.transforms import functional as TF
from tqdm.auto import tqdm

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

WORKDIR = Path.cwd()
DATASET_DIR = WORKDIR / "einstein_files"
TARGET_PATH = WORKDIR / "reference.txt"
WEIGHTS_PATH = WORKDIR / "symbol_cnn_emnist.pt"
PRED_TEXT_PATH = WORKDIR / "pred_text.txt"


DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ALPHABET = list("abcdefghijklmnopqrstuvwxyz")

image_paths = sorted(
    [p for p in DATASET_DIR.rglob("*.png")],
    key=lambda p: [int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", p.stem)],
)

if not image_paths:
    raise FileNotFoundError("Add page images under einstein_files/ first.")

print("Pages:", len(image_paths))
print("Device:", DEVICE)


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(14, 10))
axes = axes.ravel()

for ax, path in zip(axes, image_paths[:6]):
    img = Image.open(path).convert("L")
    ax.imshow(img, cmap="gray")
    ax.set_title(path.stem)
    ax.axis("off")

for ax in axes[len(image_paths[:6]):]:
    ax.axis("off")

plt.tight_layout()
plt.show()


## Предобработка и сегментация

Ниже я показываю более прямой вариант под это задание:
сначала на странице ищутся символы, а потом они просто упорядочиваются по высоте и слева направо.

Здесь используется очень простой подход:
- шум убирается через `denoise + blur + threshold`;
- символы ищутся как связные компоненты;
- потом символы группируются по высоте, чтобы сохранить порядок чтения.

Это не идеальный OCR, но для учебного кейса достаточно понятно и прозрачно.


In [ ]:
sample_path = image_paths[0]
gray = cv2.imread(str(sample_path), cv2.IMREAD_GRAYSCALE)

denoised = cv2.fastNlMeansDenoising(gray, None, h=15, templateWindowSize=7, searchWindowSize=21)
blurred = cv2.GaussianBlur(denoised, (3, 3), 0)
_, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
clean = np.zeros_like(binary)
for i in range(1, num_labels):
    if stats[i, cv2.CC_STAT_AREA] >= 8:
        clean[labels == i] = 255

num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(clean, connectivity=8)
symbol_boxes = []
for i in range(1, num_labels):
    x = int(stats[i, cv2.CC_STAT_LEFT])
    y = int(stats[i, cv2.CC_STAT_TOP])
    w = int(stats[i, cv2.CC_STAT_WIDTH])
    h = int(stats[i, cv2.CC_STAT_HEIGHT])
    area = int(stats[i, cv2.CC_STAT_AREA])
    if area >= 12 and w >= 3 and h >= 10:
        symbol_boxes.append((x, y, w, h))

symbol_boxes = sorted(symbol_boxes, key=lambda box: box[1] + box[3] / 2)

median_height = float(np.median([h for _, _, _, h in symbol_boxes])) if symbol_boxes else 12.0
row_threshold = max(12.0, 0.7 * median_height)

rows = []
for box in symbol_boxes:
    cy = box[1] + box[3] / 2
    if not rows:
        rows.append([box])
    else:
        last_row_center = np.mean([b[1] + b[3] / 2 for b in rows[-1]])
        if abs(cy - last_row_center) <= row_threshold:
            rows[-1].append(box)
        else:
            rows.append([box])

for row in rows:
    row.sort(key=lambda box: box[0])

page_with_symbols = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
for x, y, w, h in symbol_boxes:
    cv2.rectangle(page_with_symbols, (x, y), (x + w, y + h), (255, 0, 0), 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
axes[0].imshow(gray, cmap="gray")
axes[0].set_title("Оригинал")
axes[0].axis("off")

axes[1].imshow(clean, cmap="gray")
axes[1].set_title("После очистки")
axes[1].axis("off")

axes[2].imshow(page_with_symbols)
axes[2].set_title(f"Найдено символов: {len(symbol_boxes)}")
axes[2].axis("off")
plt.tight_layout()
plt.show()

first_row = rows[0]
x1 = max(0, min(x for x, _, _, _ in first_row) - 8)
y1 = max(0, min(y for _, y, _, _ in first_row) - 6)
x2 = min(clean.shape[1], max(x + w for x, _, w, _ in first_row) + 8)
y2 = min(clean.shape[0], max(y + h for _, y, _, h in first_row) + 6)
first_row_view = cv2.cvtColor(clean[y1:y2, x1:x2], cv2.COLOR_GRAY2RGB)

for x, y, w, h in first_row:
    cv2.rectangle(first_row_view, (x - x1, y - y1), (x + w - x1, y + h - y1), (255, 0, 0), 1)

plt.figure(figsize=(14, 2.5))
plt.imshow(first_row_view)
plt.title(f"Первая группа символов, найдено символов: {len(first_row)}")
plt.axis("off")
plt.show()


## Обучение модели букв

Здесь используется внешний размеченный датасет `EMNIST Letters`.

Я не усложняю:
- маленькая CNN;
- небольшой `train/val/test`;
- 5 эпох;
- сохранение лучших весов в один файл.


In [ ]:
emnist_transform = transforms.Compose(
    [
        transforms.Lambda(lambda img: TF.rotate(TF.hflip(img), 90)),
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ]
)

train_full = datasets.EMNIST(
    root=WORKDIR / "emnist",
    split="letters",
    train=True,
    download=True,
    transform=emnist_transform,
    target_transform=lambda y: y - 1,
)

test_full = datasets.EMNIST(
    root=WORKDIR / "emnist",
    split="letters",
    train=False,
    download=True,
    transform=emnist_transform,
    target_transform=lambda y: y - 1,
)

perm_train = torch.randperm(len(train_full), generator=torch.Generator().manual_seed(SEED)).tolist()
perm_test = torch.randperm(len(test_full), generator=torch.Generator().manual_seed(SEED)).tolist()

train_dataset = Subset(train_full, perm_train[:30000])
val_dataset = Subset(train_full, perm_train[30000:35000])
test_dataset = Subset(test_full, perm_test[:5000])

train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True, num_workers=2, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)
test_loader = DataLoader(test_dataset, batch_size=256, shuffle=False, num_workers=2, pin_memory=True)


class SymbolCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(128 * 3 * 3, 128),
            nn.ReLU(),
            nn.Linear(128, 26),
        )

    def forward(self, x):
        x = self.features(x)
        return self.classifier(x)


model = SymbolCNN().to(DEVICE)
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

if WEIGHTS_PATH.exists():
    model.load_state_dict(torch.load(WEIGHTS_PATH, map_location=DEVICE))
    print("Веса загружены из файла")
else:
    best_val_acc = -1.0
    best_state = None

    for epoch in range(1, 6):
        model.train()
        train_loss_sum = 0.0

        for images, targets in tqdm(train_loader, desc=f"Train epoch {epoch}/5"):
            images = images.to(DEVICE)
            targets = targets.to(DEVICE)

            optimizer.zero_grad()
            logits = model(images)
            loss = criterion(logits, targets)
            loss.backward()
            optimizer.step()

            train_loss_sum += loss.item() * len(images)

        model.eval()
        val_loss_sum = 0.0
        val_preds = []
        val_targets = []

        with torch.inference_mode():
            for images, targets in val_loader:
                images = images.to(DEVICE)
                targets = targets.to(DEVICE)
                logits = model(images)
                loss = criterion(logits, targets)

                val_loss_sum += loss.item() * len(images)
                val_preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
                val_targets.extend(targets.numpy().tolist())

        train_loss = train_loss_sum / len(train_loader.dataset)
        val_loss = val_loss_sum / len(val_loader.dataset)
        val_acc = accuracy_score(val_targets, val_preds)

        print(
            f"Epoch {epoch}: "
            f"train_loss={train_loss:.4f}, "
            f"val_loss={val_loss:.4f}, "
            f"val_acc={val_acc:.4f}"
        )

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

    model.load_state_dict(best_state)
    torch.save(model.state_dict(), WEIGHTS_PATH)

model.eval()
test_preds = []
test_targets = []

with torch.inference_mode():
    for images, targets in test_loader:
        images = images.to(DEVICE)
        logits = model(images)
        test_preds.extend(logits.argmax(dim=1).cpu().numpy().tolist())
        test_targets.extend(targets.numpy().tolist())

test_acc = accuracy_score(test_targets, test_preds)
print(f"EMNIST test accuracy = {test_acc:.4f}")


## Финальная функция OCR

Здесь всё собрано в одной функции `process(...)`, как и требуется в задании.
Сначала на странице ищутся символы, потом они группируются по высоте для порядка чтения, и только после этого модель распознаёт буквы.


In [ ]:
inference_transform = transforms.Compose(
    [
        transforms.ToTensor(),
        transforms.Normalize((0.1307,), (0.3081,)),
    ]
)


def process(einstein_files, model):
    page_paths = sorted(
        [p for p in Path(einstein_files).rglob("*.png")],
        key=lambda p: [int(x) if x.isdigit() else x.lower() for x in re.split(r"(\d+)", p.stem)],
    )

    page_texts = []

    for image_path in tqdm(page_paths, desc="OCR pages"):
        gray = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)

        denoised = cv2.fastNlMeansDenoising(gray, None, h=15, templateWindowSize=7, searchWindowSize=21)
        blurred = cv2.GaussianBlur(denoised, (3, 3), 0)
        _, binary = cv2.threshold(blurred, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)

        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(binary, connectivity=8)
        clean = np.zeros_like(binary)
        for i in range(1, num_labels):
            if stats[i, cv2.CC_STAT_AREA] >= 8:
                clean[labels == i] = 255

        num_labels, labels, stats, _ = cv2.connectedComponentsWithStats(clean, connectivity=8)
        symbol_boxes = []
        for i in range(1, num_labels):
            x = int(stats[i, cv2.CC_STAT_LEFT])
            y = int(stats[i, cv2.CC_STAT_TOP])
            w = int(stats[i, cv2.CC_STAT_WIDTH])
            h = int(stats[i, cv2.CC_STAT_HEIGHT])
            area = int(stats[i, cv2.CC_STAT_AREA])
            if area >= 12 and w >= 3 and h >= 10:
                symbol_boxes.append((x, y, w, h))

        if not symbol_boxes:
            page_texts.append("")
            continue

        symbol_boxes = sorted(symbol_boxes, key=lambda box: box[1] + box[3] / 2)
        median_height = float(np.median([h for _, _, _, h in symbol_boxes]))
        row_threshold = max(12.0, 0.7 * median_height)

        rows = []
        for box in symbol_boxes:
            cy = box[1] + box[3] / 2
            if not rows:
                rows.append([box])
            else:
                last_row_center = np.mean([b[1] + b[3] / 2 for b in rows[-1]])
                if abs(cy - last_row_center) <= row_threshold:
                    rows[-1].append(box)
                else:
                    rows.append([box])

        page_lines = []
        for row in rows:
            row = sorted(row, key=lambda box: box[0])
            median_width = float(np.median([w for _, _, w, _ in row]))
            symbol_images = []

            for x, y, w, h in row:
                sx1 = max(0, x - 2)
                sy1 = max(0, y - 2)
                sx2 = min(clean.shape[1], x + w + 2)
                sy2 = min(clean.shape[0], y + h + 2)
                crop = clean[sy1:sy2, sx1:sx2]

                coords = cv2.findNonZero(crop)
                canvas = np.zeros((28, 28), dtype=np.uint8)
                if coords is not None:
                    bx, by, bw, bh = cv2.boundingRect(coords)
                    crop = crop[by:by + bh, bx:bx + bw]
                    scale = 20 / max(bh, bw)
                    new_w = max(1, int(round(bw * scale)))
                    new_h = max(1, int(round(bh * scale)))
                    resized = cv2.resize(
                        crop,
                        (new_w, new_h),
                        interpolation=cv2.INTER_AREA if scale < 1 else cv2.INTER_LINEAR,
                    )
                    ox = (28 - new_w) // 2
                    oy = (28 - new_h) // 2
                    canvas[oy:oy + new_h, ox:ox + new_w] = resized

                symbol_images.append(canvas)

            batch = torch.stack(
                [inference_transform(Image.fromarray(img)) for img in symbol_images]
            ).to(DEVICE)

            with torch.inference_mode():
                logits = model(batch)
                preds = logits.argmax(dim=1).cpu().numpy()

            letters = [ALPHABET[int(idx)] for idx in preds]

            pieces = []
            prev_right = None
            for (x, y, w, h), letter in zip(row, letters):
                if prev_right is not None and x - prev_right > 0.9 * max(1.0, median_width):
                    pieces.append(" ")
                pieces.append(letter)
                prev_right = x + w

            line_text = "".join(pieces).strip()
            if line_text:
                page_lines.append(line_text)

        page_texts.append("\n".join(page_lines))

    pred_text = "\n".join(page_texts).lower()
    pred_text = re.sub(r"\s+", " ", pred_text).strip()

    with open(PRED_TEXT_PATH, "w", encoding="utf-8") as file:
        file.write(pred_text)

    return pred_text


In [ ]:
pred_text = process(DATASET_DIR, model)
print(pred_text[:3000])


## Финальная оценка

`top_secret.txt` используется только здесь.


In [ ]:
if TARGET_PATH.is_file():
    reference = TARGET_PATH.read_text(encoding="utf-8").lower()
    cer = distance(pred_text, reference) / max(len(reference), 1)
    print(f"CER = {cer:.4f}")
else:
    print("Add reference.txt to calculate the character error rate.")
